<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 5 · Multi-Asset Models and Basket Options

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/05_dxquant_multi_asset_and_baskets.ipynb)


## Notebook Goals

This notebook demonstrates multi-asset valuation in DXQuant:

- simulate correlated risk factors with multi-asset GBM,
- price a basket call option by Monte Carlo simulation, and
- validate basic behavior under deterministic parameter choices.


Sampling errors use independent paths or antithetic pair means. A single moment-matched batch reports `NaN` for its error; use independent seeded replications to measure batch dispersion. A shared `scenario_set` explicitly asserts ordered joint scenarios for portfolio payoff risk; it does not construct correlations.


## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

## 1. Correlated Multi-Asset GBM

A multi-asset GBM couples log-normal dynamics via a constant correlation matrix.

In [ ]:
pricing_date = dt.date(2024, 1, 1)
maturity = dt.date(2025, 1, 1)

spots = np.array([100.0, 80.0])
rate = 0.01

drifts = [rate, rate]
vols = [0.20, 0.30]
corr = np.array([[1.0, 0.5], [0.5, 1.0]])

env = dx.MarketEnvironment(name="multi", pricing_date=pricing_date)
env.add_curve("discount", dx.ConstantShortRateCurve("disc", pricing_date, rate))

process = dx.MultiAssetGeometricBrownianMotion(
    drifts=drifts,
    volatilities=vols,
    correlation=corr,
    seed=123,
    moment_matching=False,
)

pricer = dx.MonteCarloPricer(
    env=env,
    process=process,
    discount_curve_key="discount",
    paths=30000,
    steps=96,
)

pricer

## 2. Basket Call Pricing

A basket call depends on a weighted terminal sum of underlying prices.

In [ ]:
weights = [0.5, 0.5]
strike = 90.0
basket_call = dx.BasketCallPayoff(weights=weights, strike=strike)

result = pricer.price_terminal(basket_call, maturity=maturity, spot=spots)
print(f"Basket call price:   {result.price:.6f}")
print(f"Std. error:          {result.std_error:.6f}")

## 3. Deterministic Sanity Check

With zero volatilities and zero drift, the terminal basket is deterministic. The price reduces
to discounted intrinsic value.

In [ ]:
drifts_det = [0.0, 0.0]
vols_det = [0.0, 0.0]
process_det = dx.MultiAssetGeometricBrownianMotion(
    drifts=drifts_det,
    volatilities=vols_det,
    correlation=np.eye(2),
    seed=1,
)

env_det = dx.MarketEnvironment(name="det", pricing_date=pricing_date)
env_det.add_curve(
    "discount", dx.ConstantShortRateCurve("disc", pricing_date, 0.0)
)

pricer_det = dx.MonteCarloPricer(
    env=env_det,
    process=process_det,
    discount_curve_key="discount",
    paths=1,
    steps=4,
)

spot_det = np.array([100.0, 80.0])
strike_det = 90.0
basket_det = dx.BasketCallPayoff(weights=[0.5, 0.5], strike=strike_det)

res_det = pricer_det.price_terminal(
    basket_det, maturity=maturity, spot=spot_det
)
intrinsic = max(0.5 * (100.0 + 80.0) - strike_det, 0.0)

print("MC price:", res_det.price)
print("Intrinsic:", intrinsic)

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">